# 🚕 NYC Yellow Taxi Trip Data Ingestion

---

Welcome to the **NYC TAXI DATA PIPELINE - 2026**  
This pipeline ingests, audits, and loads the latest *Yellow Taxi* trips into the bronze layer.

---

> **Volume Source**  
> `/Volumes/nyctaxi/landing/operational/2026/YellowTaxi/`

---

- 🗂️ Data is loaded with **file name** and **load timestamp**  
- 💽 Written to: `NYCTAXI.BRONZE.YELLOW_TAXI`  
- 🟢 *Status*: *Ready for next processing step*

---

#### INGEST `TAXI TRIP FROM VOLUME`
- VOLUME = `/Volumes/nyctaxi/landing/operational/2026/`

In [0]:
from datetime import datetime

# =====================================================
# CALCULATE START TIME
# =====================================================

load_start_time = datetime.now()

In [0]:
from pyspark.sql.functions import input_file_name, col, current_timestamp

yellow_taxi_df = (spark.read.format('parquet')
                      .load('/Volumes/nyctaxi/landing/yellow_taxi/raw/2026/*')
                      .withColumn('file_name', col('_metadata.file_path'))
                      .withColumn('load_timestamp', current_timestamp())
                )
display(yellow_taxi_df.limit(1))

In [0]:
# LOAD THE RESULTANT DF INTO TEMP VIEW
yellow_taxi_df.createOrReplaceTempView('yellow_taxi_temp_vw')

#### LOAD CURRENT YEAR YELLOW TAXI TRIP TO
- NYCTAXI.BRONZE.YELLOW_TAXI

In [0]:
%sql
SELECT MAX(load_timestamp) AS max_load_ts
        FROM NYCTAXI.BRONZE.YELLOW_TAXI
        WHERE file_name not like '%2025%';

MERGE INTO NYCTAXI.BRONZE.YELLOW_TAXI tgt
USING (SELECT * FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, file_name ORDER BY load_timestamp DESC) as rn FROM yellow_taxi_temp_vw) WHERE rn = 1) src
ON tgt.VendorID = src.VendorID
   AND tgt.tpep_pickup_datetime = src.tpep_pickup_datetime
   AND tgt.tpep_dropoff_datetime = src.tpep_dropoff_datetime
   AND tgt.file_name = src.file_name

WHEN MATCHED THEN
  UPDATE SET
    passenger_count = src.passenger_count,
    trip_distance = src.trip_distance,
    RatecodeID = src.RatecodeID,
    store_and_fwd_flag = src.store_and_fwd_flag,
    PULocationID = src.PULocationID,
    DOLocationID = src.DOLocationID,
    payment_type = src.payment_type,
    fare_amount = src.fare_amount,
    extra = src.extra,
    mta_tax = src.mta_tax,
    tip_amount = src.tip_amount,
    tolls_amount = src.tolls_amount,
    improvement_surcharge = src.improvement_surcharge,
    total_amount = src.total_amount,
    congestion_surcharge = src.congestion_surcharge,
    Airport_fee = src.Airport_fee,
    cbd_congestion_fee = src.cbd_congestion_fee,
    file_name = src.file_name,
    load_timestamp = src.load_timestamp

WHEN NOT MATCHED THEN
  INSERT (
    VendorID,
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    passenger_count,
    trip_distance,
    RatecodeID,
    store_and_fwd_flag,
    PULocationID,
    DOLocationID,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    improvement_surcharge,
    total_amount,
    congestion_surcharge,
    Airport_fee,
    cbd_congestion_fee,
    file_name,
    load_timestamp
  )
  VALUES (
    src.VendorID,
    src.tpep_pickup_datetime,
    src.tpep_dropoff_datetime,
    src.passenger_count,
    src.trip_distance,
    src.RatecodeID,
    src.store_and_fwd_flag,
    src.PULocationID,
    src.DOLocationID,
    src.payment_type,
    src.fare_amount,
    src.extra,
    src.mta_tax,
    src.tip_amount,
    src.tolls_amount,
    src.improvement_surcharge,
    src.total_amount,
    src.congestion_surcharge,
    src.Airport_fee,
    src.cbd_congestion_fee,
    src.file_name,
    src.load_timestamp
  );

#### DESIGN OBSERVABILTY METRICES TO CAPTURE PIPELINE LOGGING MECHANISM
- `NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG`

In [0]:
######
### PERFORM AUDIT LOGGING
######  

from datetime import datetime
from pyspark.sql.functions import lit
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    TimestampType
)

# =====================================================
# WORKFLOW PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("event_type", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("status", "")
dbutils.widgets.text("message", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

# =====================================================
# Retrieve Parameters
# =====================================================

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT DATE
# =====================================================

try:
    if not raw_event_time or raw_event_time.startswith("{{"):
        event_time = datetime.now().date()
    else:
        event_time = datetime.fromisoformat(raw_event_time).date()
except:
    event_time = datetime.now().date()

# =====================================================
# RUNTIME METRICES
# =====================================================

user_name = spark.sql(
    "SELECT current_user()"
).collect()[0][0]

# =====================================================
# SOURCE SCHEMA
# =====================================================

nyc_taxi_schema = StructType([

    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
    StructField("file_name", StringType(), True),
    StructField("load_timestamp", TimestampType(), True)
])

# =====================================================
# MAIN LOAD LOGIC
# =====================================================

try:
    record_count = yellow_taxi_df.count()

    status = "SUCCESS"
    event_type = "LOAD_SUCCESS"
    message = f"Loaded {record_count} records into {target_table}"

except Exception as e:

    record_count = 0
    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)

# =====================================================
# CALCULATE END TIME
# =====================================================

load_end_time = datetime.now()

# =====================================================
# AUDIT RECORDS
# =====================================================

audit_df = spark.createDataFrame(
    [(
        log_id,
        run_id,
        event_time,
        event_type,
        source_table,
        target_table,
        record_count,
        status,
        message,
        user_name,
        notebook_path,
        pipeline_name,
        load_start_time,
        load_end_time
    )],
    [
        "log_id",
        "run_id",
        "event_time",
        "event_type",
        "source_table",
        "target_table",
        "record_count",
        "status",
        "message",
        "user_name",
        "notebook_path",
        "pipeline_name",
        "load_start_time",
        "load_end_time"
    ]
)

audit_df = (audit_df
        .withColumn("event_time", lit(event_time).cast("date"))
        .withColumn("load_start_time", lit(load_start_time).cast("timestamp"))
        .withColumn("load_end_time", lit(load_end_time).cast("timestamp"))
)

# =====================================================
# WRITE TO AUDIT TABLE
# =====================================================

audit_df.write \
    .mode("append") \
    .saveAsTable("NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG")

In [0]:
%sql
-- VALIDATE THE RECORDS
SELECT 
TIMESTAMPDIFF(
           SECOND,
           load_start_time,
           load_end_time
       ) / 60.0 AS load_duration_minutes,
       *
FROM NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG;

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.BRONZE.YELLOW_TAXI')